# Capítulo 1 — Dados e Tabelas: Um lugar para cada coisa

**Livro:** começa na página **37** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(1, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## Tome o comando! · PDF p. 52

#### Exercício: Onde estou?

No MySQL o livro usa `CREATE DATABASE` e `USE`. Aqui cada capítulo tem um **esquema** próprio (cap01, cap02...), que funciona como um banco separado. Liste os esquemas que existem e mostre em qual você está (`current_schema()`).

In [ ]:
%%sql
SELECT schema_name, schema_name = current_schema() AS estou_aqui
FROM information_schema.schemata
WHERE schema_name NOT LIKE 'pg_%' AND schema_name <> 'information_schema'
ORDER BY schema_name;

## Preparando a tabela: o comando CREATE TABLE · PDF p. 54

#### Exercício: Sua primeira tabela

Crie a tabela `lista_compras` com as colunas `item` (texto de até 30 caracteres) e `quantidade` (número inteiro).

In [ ]:
%%sql
CREATE TABLE lista_compras (
    item VARCHAR(30),
    quantidade INT
);

## Criando uma tabela um pouco mais complicada · PDF p. 55

#### Exercício: Escolhendo os tipos

Crie `receitas` com: `nome` (até 40), `tempo_preparo` (minutos, inteiro), `porcoes` (inteiro), `custo` (dinheiro, 2 casas: `DECIMAL(6,2)`), `vegetariana` (`CHAR(1)`: S/N), `criada_em` (`DATE`) e `modo_preparo` (`TEXT`, texto longo).

In [ ]:
%%sql
CREATE TABLE receitas (
    nome VARCHAR(40),
    tempo_preparo INT,
    porcoes INT,
    custo DECIMAL(6,2),
    vegetariana CHAR(1),
    criada_em DATE,
    modo_preparo TEXT
);

## Finalmente, criando a tabela meus_contatos · PDF p. 57

#### Exercício: A tabela meus_contatos

Crie `meus_contatos` com: `sobrenome`, `primeiro_nome`, `email`, `sexo` (`CHAR(1)`), `aniversario` (`DATE`), `profissao`, `local`, `estado_civil`, `interesses` e `procurando`. Escolha tamanhos sensatos.

In [ ]:
%%sql
CREATE TABLE meus_contatos (
    sobrenome VARCHAR(30),
    primeiro_nome VARCHAR(20),
    email VARCHAR(50),
    sexo CHAR(1),
    aniversario DATE,
    profissao VARCHAR(50),
    local VARCHAR(50),
    estado_civil VARCHAR(20),
    interesses VARCHAR(100),
    procurando VARCHAR(100)
);

## Sua tabela DESCrita · PDF p. 62

#### Exercício: O DESC do PostgreSQL

O `DESC meus_contatos` do MySQL não existe aqui. Consulte `information_schema.columns` para ver coluna, tipo, tamanho, se aceita nulo e o valor padrão.

In [ ]:
%%sql
SELECT column_name AS coluna, data_type AS tipo, character_maximum_length AS tamanho,
       is_nullable AS aceita_nulo, column_default AS padrao
FROM information_schema.columns
WHERE table_schema = current_schema() AND table_name = 'meus_contatos'
ORDER BY ordinal_position;

## Você não pode recriar uma tabela ou banco de dados já existente! · PDF p. 64

#### Exercício: Criar de novo dá erro

Tente criar `lista_compras` outra vez (dentro de `try/except` em Python, com `%sql` na frente do comando) e veja o erro. Depois use `CREATE TABLE IF NOT EXISTS`, que não reclama.

In [ ]:
try:
    %sql CREATE TABLE lista_compras (item VARCHAR(30), quantidade INT)
except Exception as erro:
    print("Erro esperado:", str(erro).splitlines()[-2:])
%sql CREATE TABLE IF NOT EXISTS lista_compras (item VARCHAR(30), quantidade INT)

## Fora com a tabela antiga, pra dentro com a nova · PDF p. 65

#### Exercício: DROP TABLE e recriar

Apague `lista_compras` com `DROP TABLE` e crie de novo com uma coluna a mais: `unidade` (até 10 caracteres). Mostre a estrutura nova.

In [ ]:
%%sql
DROP TABLE lista_compras;
CREATE TABLE lista_compras (
    item VARCHAR(30),
    quantidade INT,
    unidade VARCHAR(10)
);
SELECT column_name AS coluna, data_type AS tipo, character_maximum_length AS tamanho,
       is_nullable AS aceita_nulo, column_default AS padrao
FROM information_schema.columns
WHERE table_schema = current_schema() AND table_name = 'lista_compras'
ORDER BY ordinal_position;

## Criando o comando INSERT · PDF p. 69

#### Exercício: INSERT com todas as colunas

Insira em `meus_contatos` uma pessoa (invente os dados) listando as 10 colunas e os 10 valores na mesma ordem. Datas vão como `'AAAA-MM-DD'`.

In [ ]:
%%sql
INSERT INTO meus_contatos
    (sobrenome, primeiro_nome, email, sexo, aniversario, profissao, local, estado_civil, interesses, procurando)
VALUES
    ('Moura', 'Marina', 'marina.m@exemplo.com', 'F', '1989-06-11', 'Analista de Dados', 'Recife, PE',
     'solteira', 'trilhas, fotografia', 'amizade');

## Variações para o comando INSERT · PDF p. 73

#### Exercício: Três variações

Faça 3 inserções: (a) com as colunas em outra ordem; (b) só com `primeiro_nome` e `email` (as outras ficam nulas); (c) duas pessoas num único `INSERT` (várias linhas em `VALUES`).

In [ ]:
%%sql
INSERT INTO meus_contatos (primeiro_nome, sobrenome, local, sexo)
VALUES ('Lucas', 'Lins', 'Maceió, AL', 'M');
INSERT INTO meus_contatos (primeiro_nome, email) VALUES ('Iara', 'iara@exemplo.com');
INSERT INTO meus_contatos (primeiro_nome, sobrenome, profissao) VALUES
    ('Bruno', 'Barbosa', 'Professor'),
    ('Carla', 'Cardoso', 'Designer');

## Espie sua tabela com o comando SELECT · PDF p. 75

#### Exercício: SELECT * e os NULLs

Mostre tudo de `meus_contatos`. Repare nas colunas vazias (`None`/`NULL`) de quem foi inserido sem todos os valores.

In [ ]:
%%sql
SELECT * FROM meus_contatos;

## Controlando seu NULL interior · PDF p. 76

#### Exercício: NOT NULL

Crie `rosquinhas` com `nome` e `tipo`, ambos `NOT NULL`. Insira uma válida e tente (com `try/except`) inserir uma sem `tipo`.

In [ ]:
%sql CREATE TABLE rosquinhas (nome VARCHAR(20) NOT NULL, tipo VARCHAR(10) NOT NULL)
%sql INSERT INTO rosquinhas VALUES ('Coco', 'assada')
try:
    %sql INSERT INTO rosquinhas (nome) VALUES ('Doce de leite')
except Exception as erro:
    print("Bloqueado pelo NOT NULL:", "not-null" in str(erro) or "null value" in str(erro))
%sql SELECT * FROM rosquinhas

## NOT NULL aparece em DESC · PDF p. 78

#### Exercício: Conferindo o NOT NULL

Veja a estrutura de `rosquinhas`: a coluna `aceita_nulo` deve dizer `NO`.

In [ ]:
%%sql
SELECT column_name AS coluna, data_type AS tipo, character_maximum_length AS tamanho,
       is_nullable AS aceita_nulo, column_default AS padrao
FROM information_schema.columns
WHERE table_schema = current_schema() AND table_name = 'rosquinhas'
ORDER BY ordinal_position;

## Preencha os espaços em branco com DEFAULT · PDF p. 79

#### Exercício: Valor padrão

Crie `doces` com `nome` (NOT NULL) e `preco` `DECIMAL(4,2) NOT NULL DEFAULT 1.00`. Insira um doce sem preço e outro com preço, e mostre a tabela.

In [ ]:
%%sql
CREATE TABLE doces (nome VARCHAR(20) NOT NULL, preco DECIMAL(4,2) NOT NULL DEFAULT 1.00);
INSERT INTO doces (nome) VALUES ('Paçoca');
INSERT INTO doces (nome, preco) VALUES ('Cocada', 2.50);
SELECT * FROM doces;